# 02. Loading Engineering Datasets
**Engineering Data Processing with Python** | one-day course | approx. 50 minutes

> **EXERCISE NOTEBOOK.** Work top to bottom. Run every cell in order (Shift+Enter). Where you see `____` replace it with your answer. Where you see `# TODO`, write the code. If an error mentions `____` (for example `NameError: name '____' is not defined`), you have a blank left to fill. Blanks are four underscores.


In [ ]:
# ===================== SETUP: run this cell first (click it, then Shift+Enter) =====================
# It makes sure the course data is available in this Colab session and loads the tools we need.
import os, zipfile, urllib.request
from pathlib import Path

DATA_ZIP_URL = ""   # if your trainer gives you a download link for data.zip, paste it between the quotes

def get_course_data():
    if Path("data").is_dir():
        print("Data folder found:", sorted(p.name for p in Path("data").iterdir())[:20])
        return
    if not Path("data.zip").exists() and DATA_ZIP_URL:
        print("Downloading course data...")
        urllib.request.urlretrieve(DATA_ZIP_URL, "data.zip")
    zip_path = Path("data.zip")
    if not zip_path.exists():
        try:
            from google.colab import files
            print("Choose data.zip from your computer (your trainer will tell you where it is).")
            uploaded = files.upload()
            zip_path = Path(next(iter(uploaded)))
        except ImportError:
            raise FileNotFoundError("Put data.zip or the data folder next to this notebook.")
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(".")
    print("Data ready:", sorted(p.name for p in Path("data").iterdir()))

get_course_data()
Path("outputs").mkdir(exist_ok=True)

import pandas as pd
import numpy as np
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

def check(condition, ok="Looks right.", hint="Not quite yet. Re-read the instructions above the cell."):
    """Tiny self-check helper used throughout the course."""
    print(("PASS: " + ok) if condition else ("NOT YET: " + hint))

## What you will do

* Load a large CSV of energy data from a steel plant and parse its dates **correctly**.
* Find and fix a timestamp fault that no error message will ever tell you about.
* Load Excel files with title rows, multiple sheets and two-row headers.
* Load the Kilbrack work orders with the right settings for missing values and dates.

The general lesson: `pd.read_csv()` and `pd.read_excel()` will happily load almost anything. Loading without an error
does not mean loading correctly. You check.

## 1. Steel plant energy data (35,000 rows of 15-minute readings)

### Exercise 2.1 [Level 1: fill the blanks]

In [ ]:
steel = pd.read_csv("data/____")
print(steel.shape)
print(steel.____)
steel.head()

`date` is `object` (text). Until it is a real datetime you cannot sort by time properly, resample to daily totals
or calculate durations.

### The date order trap

Irish and UK exports usually write day/month/year. pandas, left to guess, often assumes month/day/year.
Watch what happens to the 1st of February:

In [ ]:
sample = pd.Series(["01/02/2018 00:15", "13/02/2018 00:15"])
print(pd.to_datetime(sample.iloc[:1]))                     # guessed: becomes 2 January
print(pd.to_datetime(sample, format="%d/%m/%Y %H:%M"))     # told explicitly: 1 and 13 February

The first guess is wrong and there is **no error**. Day 13 would have exposed it, but days 1 to 12 parse without complaint in
the wrong month. Always give an explicit `format` when you know it.

Format codes: `%d` day, `%m` month, `%Y` 4-digit year, `%H` hour (24h), `%M` minute, `%S` second.

### Exercise 2.2 [Level 1: fill the blanks]

In [ ]:
steel["date"] = pd.to_datetime(steel["date"], format="____")
print(steel["date"].dtype)
print("From", steel["date"].min(), "to", steel["date"].max())

In [ ]:
check(str(steel["date"].dtype).startswith("datetime64") and steel["date"].dt.month.nunique() == 12,
      "Dates parsed, all 12 months present.", "Check the format string: day first, then month.")

### A fault nobody will tell you about

Time series should go forwards. Let's check.

In [ ]:
print("Is the time column always increasing?", steel["date"].is_monotonic_increasing)

steps = steel["date"].diff()                 # time between each row and the one before
backwards = steel[steps < pd.Timedelta(0)]   # rows where time went backwards
print("Rows where time goes backwards:", len(backwards))
steel.loc[94:97, ["date", "Usage_kWh", "NSM", "Day_of_week"]]

Look at rows 94 to 96. After `2018-01-01 23:45` comes `2018-01-01 00:00`, then `2018-01-02 00:15`.
The midnight reading is stamped with the **wrong day**: it belongs to the start of the *next* day.
The source system wrote the end-of-day reading with today's date. This happens once a day, 365 times.

`NSM` (number of seconds from midnight) is 0 on that row, which supports the diagnosis. This is how you investigate:
find the anomaly, look at the neighbours, use another column to confirm.

### Exercise 2.3 [Level 2: finish the code]

Fix it: every row whose time is exactly 00:00 should move forward one day.

* `steel["date"].dt.hour` and `steel["date"].dt.minute` give the hour and minute.
* Build a True/False *mask* for "hour is 0 **and** minute is 0" (use `&` and brackets around each condition).
* Add `pd.Timedelta(days=1)` to those rows using `steel.loc[mask, "date"]`.

In [ ]:
# TODO: build the mask of midnight rows
midnight = None   # replace None with your condition

# TODO: add one day to the date on those rows (steel.loc[midnight, "date"] = ...)


print("Monotonic now?", steel["date"].is_monotonic_increasing)
print("Last timestamp:", steel["date"].max())

In [ ]:
gaps = steel["date"].diff().dropna()
check(steel["date"].is_monotonic_increasing and steel["date"].max() == pd.Timestamp("2019-01-01 00:00")
      and (gaps == pd.Timedelta(minutes=15)).all(),
      "Every step is exactly 15 minutes and the year ends at 2019-01-01 00:00.",
      "Not fixed yet. Did you add one day to the midnight rows only?")

## 2. Quick profiling of the steel data

### Exercise 2.4 [Level 1: fill the blanks]
Daily energy use. `resample("D")` groups a time series by day (like a pivot table on dates).

*A detail that matters:* each reading is stamped at the **end** of its 15-minute interval, so the 00:00 reading
belongs to the day before. `closed="right", label="left"` tells pandas that. Skip it and every daily total is
shifted by one reading. For a utility bill reconciliation, that matters.

In [ ]:
daily_kwh = (steel.set_index("____")["Usage_kWh"]
                  .resample("D", closed="right", label="left")
                  .____())
print(len(daily_kwh), "days")
daily_kwh.plot(title="Daily energy use (kWh)", figsize=(10, 3));

In [ ]:
check(len(daily_kwh) == 365, "365 days. The weekly pattern (weekends lower) is visible.", "Expected 365 days.")

In [ ]:
steel["Load_Type"].value_counts()

## 3. Excel with title rows: the Kilbrack asset register

### Exercise 2.5 [Level 1: fill the blanks]
First, load it the naive way and look at the column names.

In [ ]:
naive = pd.read_excel("data/asset_register.xlsx")
naive.head(6)

The real header is on the 4th row of the sheet. Above it are a title and an export note. Tell pandas to skip them.

In [ ]:
register = pd.read_excel("data/asset_register.xlsx", skiprows=____)
print(register.columns.tolist())
register.head()

In [ ]:
check("Asset ID" in register.columns, "Header found.", "Expected a column called 'Asset ID'. How many rows are above the header?")

Excel files often have more than one sheet. `sheet_name=None` loads **all** sheets into a dictionary
(sheet name: DataFrame).

In [ ]:
all_sheets = pd.read_excel("data/asset_register.xlsx", sheet_name=None, skiprows=____)
print(list(all_sheets.____()))

Careful: `skiprows=3` applied to *every* sheet, but the `Decommissioned` sheet has its header on row 1.
Load that one on its own.

In [ ]:
decommissioned = pd.read_excel("data/asset_register.xlsx", sheet_name="____")
decommissioned

In [ ]:
register.dtypes

`Install Date` and `Rated Power` are `object`. Some cells are real Excel dates, some are text in different formats,
and power arrives as `38 kW`, `38kW`, `38000 W` and plain `38`. We fix those in Module 03.

## 4. Excel with a two-row header: thermocouple data from casting trials

`thermal-profiles.xlsx` holds cooling curves for three aluminium-silicon alloys. Each sheet has a title, a blank row,
a row of thermocouple **positions** (mm) and then the real header. Have a look:

In [ ]:
pd.read_excel("data/thermal-profiles.xlsx", sheet_name="Al-11wt.%Si", header=None, nrows=6)

### Exercise 2.6 [Level 2: finish the code]

1. Load the sheet `Al-11wt.%Si` using row 4 as the header (`header=3`, counting from zero).
2. Read the positions row separately (`header=None, nrows=3`, then the last row).
3. Rename columns to `time_s`, then `tc1_4mm`, `tc2_15mm`, and so on, so the position travels with the data.

In [ ]:
SHEET = "Al-11wt.%Si"
thermal = pd.read_excel("data/thermal-profiles.xlsx", sheet_name=SHEET, header=3)

top = pd.read_excel("data/thermal-profiles.xlsx", sheet_name=SHEET, header=None, nrows=3)
positions_mm = top.iloc[2, 1:].tolist()     # row 3 of the sheet, every column except the first
print("Positions (mm):", positions_mm)

# TODO: build a list of new column names: "time_s" first, then f"tc{i}_{int(pos)}mm" for each position
#       (i counts from 1; int() because Excel gives 4.0, not 4). A for loop with enumerate(positions_mm, start=1) works well.
new_cols = ["time_s"]

# TODO: assign new_cols to thermal.columns (only once your list has the right length)

thermal.head()

In [ ]:
check(thermal.shape == (2001, 8) and thermal.columns[0] == "time_s" and thermal.columns[1] == "tc1_4mm",
      "Loaded with meaningful names.", "Expected 2001 rows, 8 columns, first two named time_s and tc1_4mm.")

In [ ]:
thermal.set_index("time_s").plot(figsize=(10, 4), title="Cooling curves, Al-11wt.%Si", ylabel="°C");

## 5. The Kilbrack work orders, loaded properly

Default settings already treat blank cells and `N/A` as missing. This export also uses `-` for "nothing".
`na_values` adds extra markers.

### Exercise 2.7 [Level 1: fill the blanks]

In [ ]:
wo = pd.read_csv("data/work_orders_cmms.csv", na_values=["____"])
print(wo.shape)
print("Missing parts cost:", wo["parts_cost_eur"].isna().sum())
wo.head()

### Exercise 2.8 [Level 2: finish the code]

Most `date_raised` values are `dd/mm/yyyy HH:MM`, but a few were entered as `yyyy-mm-dd HH:MM`.
A robust pattern for mixed formats:

1. Parse with the main format and `errors="coerce"` (anything that does not fit becomes `NaT`, "not a time").
2. Look at what failed.
3. Parse the failures with the second format and fill them in.
4. Count what is still missing. It should be zero, or you should know why.

In [ ]:
raised = pd.to_datetime(wo["date_raised"], format="%d/%m/%Y %H:%M", errors="coerce")
failed = raised.isna()
print("Did not fit the main format:", failed.sum())
wo.loc[failed, "date_raised"].head()

In [ ]:
# TODO: parse ONLY the failed rows with the format "%Y-%m-%d %H:%M"
second_try = None   # replace None

# TODO: fill the gaps in raised with second_try (hint: raised.fillna(second_try))

wo["date_raised"] = raised
print("Still missing:", wo["date_raised"].isna().sum())

In [ ]:
check(wo["date_raised"].isna().sum() == 0 and str(wo["date_raised"].dtype).startswith("datetime64"),
      "Every date parsed, and you know exactly which formats were in the file.",
      "Some dates are still missing. Did you fill raised with second_try?")

**Why not just let pandas guess?** `pd.to_datetime(col, format="mixed", dayfirst=True)` would work on this file.
It guesses each value separately. That is convenient for exploring, but in a pipeline that runs every month you
want it to **fail visibly** when a new format turns up, not guess. Explicit formats plus a count of failures
gives you that.

## 6. Stretch [Level 3: Gemini allowed]

`data/manifest.TXT` and `data/manifest.csv` both list files from a microscopy archive. The TXT is not a table, each
line looks like `diatom-01.TIF (image/tiff) 4922346 bytes.`

1. Turn the TXT into a DataFrame with columns `file_name`, `file_type`, `size_bytes` (ask Gemini for a regular
   expression, then ask it to explain each part of it).
2. Compare it with the CSV. Are the two lists the same? Is anything inconsistent inside them?

Write down what you found before looking at the solution.

In [ ]:
# TODO: your code here

## Summary

* Give `to_datetime` an explicit `format`. Day/month confusion is silent.
* Check time series are monotonic and evenly spaced. Faults hide in the order of rows.
* `skiprows`, `header`, `sheet_name` and `nrows` handle most Excel layouts.
* `na_values` adds extra "missing" markers.
* For mixed formats: parse, count failures, handle them, count again.

Next: **03 Cleaning and Standardising**.